# Stage 4.1 — WOE / IV / Binning Method Design

## Objective, authority, and real-data boundary

This Notebook freezes and transparently validates the methodology that later Stage 4 work will use. It does **not** determine where any real predictor should be cut.

All calculations below use explicitly synthetic toy counts or synthetic toy values. No project CSV, Train frame, Holdout Test frame, real predictor, or real target is loaded. The 6,000-row Holdout Test remains sealed. No real WOE/IV, supervised binning, ranking, variable selection, model fitting, cross-validation, AUC, or KS is performed.


In [1]:
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

REAL_PROJECT_DATA_LOADED = False
REAL_PREDICTOR_ANALYSIS_PERFORMED = False
HOLDOUT_TEST_USED = False
VARIABLE_SELECTION_PERFORMED = False
MODELING_PERFORMED = False
KFOLD_EXECUTED = False

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Method-design data source: synthetic toy examples only")
print("Holdout Test status: SEALED — not loaded or referenced by calculation code")


Python executable: /Users/yu/2.工作/Risk Analysis Project/.venv/bin/python
Python version: 3.9.6
Method-design data source: synthetic toy examples only
Holdout Test status: SEALED — not loaded or referenced by calculation code


## 1. Frozen Good / Bad / Event convention

For the project target `default.payment.next.month`:

- `0 = Good = Non-default = Non-event`
- `1 = Bad = Default = Event`

This direction must never be reversed later.

For bin $j$:

$$DistGood_j = \frac{Good_j}{TotalGood}$$

$$DistBad_j = \frac{Bad_j}{TotalBad}$$

$$WOE_j = \ln\left(\frac{DistGood_j}{DistBad_j}\right)$$

Under this frozen definition:

- positive WOE means relatively stronger Good representation and therefore relatively lower risk;
- negative WOE means relatively stronger Bad representation and therefore relatively higher risk;
- WOE near zero means the bin represents Good and Bad populations similarly and has weak separation.


## 2. Transparent synthetic WOE / IV toy counts

The four bins and counts below are invented solely for hand calculation. They are not derived from any project field or customer record. The main example has no zero-event bin, so no smoothing is needed.


In [2]:
toy_counts = pd.DataFrame({
    "Bin": ["Toy_A", "Toy_B", "Toy_C", "Toy_D"],
    "Good": [40, 30, 20, 10],
    "Bad": [10, 30, 25, 35],
})
toy_counts["Count"] = toy_counts["Good"] + toy_counts["Bad"]
toy_counts = toy_counts[["Bin", "Count", "Good", "Bad"]]

assert toy_counts["Good"].sum() == 100
assert toy_counts["Bad"].sum() == 100
assert (toy_counts["Count"] == toy_counts["Good"] + toy_counts["Bad"]).all()
display(toy_counts)
print("Synthetic totals — Good:", toy_counts["Good"].sum(), "Bad:", toy_counts["Bad"].sum())


,Bin,Count,Good,Bad
0,Toy_A,50,40,10
1,Toy_B,60,30,30
2,Toy_C,45,20,25
3,Toy_D,45,10,35


Synthetic totals — Good: 100 Bad: 100


## 3. Manual WOE and IV calculation

The next cell writes every vector in the formula directly. It is intentionally independent of the reusable helper introduced afterward.

$$DefaultRate_j = \frac{Bad_j}{Good_j + Bad_j}$$

$$IV_j = (DistGood_j - DistBad_j)\times WOE_j$$

$$IV = \sum_j IV_j$$


In [3]:
manual_good = np.array([40.0, 30.0, 20.0, 10.0])
manual_bad = np.array([10.0, 30.0, 25.0, 35.0])
manual_count = manual_good + manual_bad
manual_total_good = manual_good.sum()
manual_total_bad = manual_bad.sum()
manual_default_rate = manual_bad / manual_count
manual_dist_good = manual_good / manual_total_good
manual_dist_bad = manual_bad / manual_total_bad
manual_woe = np.log(manual_dist_good / manual_dist_bad)
manual_iv_contribution = (manual_dist_good - manual_dist_bad) * manual_woe
manual_total_iv = manual_iv_contribution.sum()

manual_table = pd.DataFrame({
    "Bin": toy_counts["Bin"],
    "Count": manual_count.astype(int),
    "Good": manual_good.astype(int),
    "Bad": manual_bad.astype(int),
    "Default Rate": manual_default_rate,
    "Dist Good": manual_dist_good,
    "Dist Bad": manual_dist_bad,
    "WOE": manual_woe,
    "IV Contribution": manual_iv_contribution,
})

display(manual_table)
print("Manual total IV:", f"{manual_total_iv:.12f}")


,Bin,Count,Good,Bad,Default Rate,Dist Good,Dist Bad,WOE,IV Contribution
0,Toy_A,50,40,10,0.200000,0.400000,0.100000,1.386294,0.415888
1,Toy_B,60,30,30,0.500000,0.300000,0.300000,0.000000,0.000000
2,Toy_C,45,20,25,0.555556,0.200000,0.250000,-0.223144,0.011157
3,Toy_D,45,10,35,0.777778,0.100000,0.350000,-1.252763,0.313191


Manual total IV: 0.740236228026


## 4. Programmatic helper and equality assertions

The helper accepts already aggregated synthetic Good/Bad counts. It uses the frozen Good-over-Bad WOE direction. Additive smoothing is optional and defaults to zero; the main toy example therefore remains unsmoothed.


In [4]:
def calculate_woe_iv_from_counts(counts, smoothing=0.0):
    required = {"Bin", "Good", "Bad"}
    assert required.issubset(counts.columns)
    assert smoothing >= 0

    result = counts[["Bin", "Good", "Bad"]].copy()
    result["Good"] = result["Good"].astype(float)
    result["Bad"] = result["Bad"].astype(float)
    result["Count"] = result["Good"] + result["Bad"]

    good_for_woe = result["Good"] + smoothing
    bad_for_woe = result["Bad"] + smoothing
    total_good_for_woe = good_for_woe.sum()
    total_bad_for_woe = bad_for_woe.sum()

    result["Default Rate"] = result["Bad"] / result["Count"]
    result["Dist Good"] = good_for_woe / total_good_for_woe
    result["Dist Bad"] = bad_for_woe / total_bad_for_woe
    with np.errstate(divide="ignore", invalid="ignore"):
        result["WOE"] = np.log(result["Dist Good"] / result["Dist Bad"])
        result["IV Contribution"] = (result["Dist Good"] - result["Dist Bad"]) * result["WOE"]

    if smoothing > 0:
        result.insert(3, "Good for WOE", good_for_woe)
        result.insert(5, "Bad for WOE", bad_for_woe)

    ordered = ["Bin", "Count", "Good"]
    if smoothing > 0:
        ordered.append("Good for WOE")
    ordered.append("Bad")
    if smoothing > 0:
        ordered.append("Bad for WOE")
    ordered += ["Default Rate", "Dist Good", "Dist Bad", "WOE", "IV Contribution"]
    result = result[ordered]
    return result, float(result["IV Contribution"].sum())

programmatic_table, programmatic_total_iv = calculate_woe_iv_from_counts(toy_counts)

numeric_columns = [
    "Count", "Good", "Bad", "Default Rate", "Dist Good", "Dist Bad", "WOE", "IV Contribution"
]
np.testing.assert_allclose(
    programmatic_table[numeric_columns].to_numpy(dtype=float),
    manual_table[numeric_columns].to_numpy(dtype=float),
    rtol=1e-12, atol=1e-12,
)
assert np.isclose(programmatic_total_iv, manual_total_iv, rtol=1e-12, atol=1e-12)
assert (programmatic_table["Count"] == programmatic_table["Good"] + programmatic_table["Bad"]).all()
assert np.isclose(programmatic_table["Dist Good"].sum(), 1.0, atol=1e-12)
assert np.isclose(programmatic_table["Dist Bad"].sum(), 1.0, atol=1e-12)
assert (programmatic_table["WOE"] > 0).any()
assert (programmatic_table["WOE"] < 0).any()
assert np.isclose(programmatic_table["WOE"], 0.0, atol=1e-12).any()

display(programmatic_table)
print("Programmatic total IV:", f"{programmatic_total_iv:.12f}")
print("Manual vs programmatic bin-level and total-IV assertions: PASS (rtol=atol=1e-12)")


,Bin,Count,Good,Bad,Default Rate,Dist Good,Dist Bad,WOE,IV Contribution
0,Toy_A,50.000000,40.000000,10.000000,0.200000,0.400000,0.100000,1.386294,0.415888
1,Toy_B,60.000000,30.000000,30.000000,0.500000,0.300000,0.300000,0.000000,0.000000
2,Toy_C,45.000000,20.000000,25.000000,0.555556,0.200000,0.250000,-0.223144,0.011157
3,Toy_D,45.000000,10.000000,35.000000,0.777778,0.100000,0.350000,-1.252763,0.313191


Programmatic total IV: 0.740236228026
Manual vs programmatic bin-level and total-IV assertions: PASS (rtol=atol=1e-12)


## 5. Default Rate versus WOE

Default Rate is an **inside-the-bin proportion**: among customers in one bin, what fraction are Bad?

WOE is a **whole-population representation comparison**: how much of all Goods falls in this bin versus how much of all Bads falls in it?

They often move in related directions, but they are not the same quantity. In the toy table, `Toy_B` has Default Rate 0.50 while WOE is exactly 0 because it contains 30% of all Goods and 30% of all Bads. `Toy_A` has positive WOE and relatively lower risk; `Toy_D` has negative WOE and relatively higher risk.


In [5]:
woe_teaching_table = programmatic_table[["Bin", "Default Rate", "Dist Good", "Dist Bad", "WOE"]].copy()
woe_teaching_table["Frozen sign interpretation"] = np.select(
    [woe_teaching_table["WOE"] > 1e-12, woe_teaching_table["WOE"] < -1e-12],
    ["Positive WOE → relatively lower risk", "Negative WOE → relatively higher risk"],
    default="WOE near 0 → weak separation",
)
display(woe_teaching_table)


,Bin,Default Rate,Dist Good,Dist Bad,WOE,Frozen sign interpretation
0,Toy_A,0.200000,0.400000,0.100000,1.386294,Positive WOE → relatively lower risk
1,Toy_B,0.500000,0.300000,0.300000,0.000000,WOE near 0 → weak separation
2,Toy_C,0.555556,0.200000,0.250000,-0.223144,Negative WOE → relatively higher risk
3,Toy_D,0.777778,0.100000,0.350000,-1.252763,Negative WOE → relatively higher risk


## 6. IV interpretation and its non-automatic role

| IV | Descriptive interpretation |
|---:|---|
| < 0.02 | Very weak |
| 0.02–0.10 | Weak |
| 0.10–0.30 | Moderate |
| 0.30–0.50 | Strong |
| > 0.50 | Very strong; investigate carefully |

These are descriptive reference ranges only. They are **not** an automatic rule such as `IV < 0.02 → drop`. The synthetic toy total IV must not be interpreted as evidence about any real predictor.

Stage 4.4 will treat IV as important screening evidence together with risk pattern, business interpretability, redundancy, multicollinearity, and later Logistic stability. Stage 4.1 performs no real-variable IV ranking, filtering, or recommendation.


## 7. Frozen continuous supervised-binning workflow

Ordinary continuous variables will use:

> **Quantile pre-binning → target-aware adjacent merging**

Policy:

1. Create at most about 20 quantile-based candidate pre-bins, aiming for reasonably balanced counts.
2. If unique values cannot support that many bins, respect the actual structure; never jitter values.
3. Treat pre-bins as candidates, not final bins.
4. Compare adjacent bins using Train-only default rates, WOE, sample size, stability, and business meaning in Stage 4.3.
5. Merge adjacent bins only. Do not search arbitrary combinations, maximize IV through repeated cut hunting, use opaque optimal-binning tools, or consult Holdout Test.

The next demonstration is synthetic. Its values, event labels, intervals, and merge have no application to UCI variables.


In [6]:
# Synthetic continuous values: 200 invented observations, five equal-frequency pre-bins.
toy_continuous = pd.DataFrame({"ToyValue": np.arange(1.0, 201.0)})
toy_bad_counts_by_block = [4, 10, 11, 20, 30]
toy_events = np.zeros(200, dtype=int)
for block_number, bad_count in enumerate(toy_bad_counts_by_block):
    block_start = block_number * 40
    toy_events[block_start:block_start + bad_count] = 1
toy_continuous["ToyEvent"] = toy_events
toy_continuous["PreBin"] = pd.qcut(toy_continuous["ToyValue"], q=5, duplicates="raise")

toy_prebin_counts = (
    toy_continuous.groupby("PreBin", observed=True)["ToyEvent"]
    .agg(Count="size", Bad="sum").reset_index()
)
toy_prebin_counts["Good"] = toy_prebin_counts["Count"] - toy_prebin_counts["Bad"]
toy_prebin_counts["Bin"] = toy_prebin_counts["PreBin"].astype(str)
toy_prebin_table, toy_prebin_total_iv = calculate_woe_iv_from_counts(
    toy_prebin_counts[["Bin", "Good", "Bad"]]
)

assert len(toy_prebin_table) == 5
assert (toy_prebin_table["Count"] == 40).all()
assert np.isclose(toy_prebin_table.loc[1, "Default Rate"], 0.25)
assert np.isclose(toy_prebin_table.loc[2, "Default Rate"], 0.275)
assert abs(toy_prebin_table.loc[1, "WOE"] - toy_prebin_table.loc[2, "WOE"]) < 0.15

display(toy_prebin_table[["Bin", "Count", "Good", "Bad", "Default Rate", "WOE"]])
print("Synthetic pre-bins 2 and 3 have close default rates and WOE; they are adjacent merge candidates.")


,Bin,Count,Good,Bad,Default Rate,WOE
0,"(0.999, 40.8]",40.000000,36.000000,4.000000,0.100000,1.686399
1,"(40.8, 80.6]",40.000000,30.000000,10.000000,0.250000,0.587787
2,"(80.6, 120.4]",40.000000,29.000000,11.000000,0.275000,0.458575
3,"(120.4, 160.2]",40.000000,20.000000,20.000000,0.500000,-0.510826
4,"(160.2, 200.0]",40.000000,10.000000,30.000000,0.750000,-1.609438


Synthetic pre-bins 2 and 3 have close default rates and WOE; they are adjacent merge candidates.


## 8. Toy adjacent-merging demonstration

The synthetic second and third pre-bins are adjacent, each has 40 observations, and their default rates are 25.0% and 27.5%. Their WOE values are also close. Conceptually merging them raises support to 80 observations and reduces separation driven by a very small risk difference.

This merge is target-aware because synthetic Good/Bad behavior informed it. It is not an automated optimum and creates no real cut point. A real final decision must also consider sample adequacy, stability, business interpretation, and reasonable risk structure.


In [7]:
toy_merged_counts = pd.DataFrame({
    "Bin": [
        toy_prebin_table.loc[0, "Bin"],
        f"{toy_prebin_table.loc[1, 'Bin']} + {toy_prebin_table.loc[2, 'Bin']}",
        toy_prebin_table.loc[3, "Bin"],
        toy_prebin_table.loc[4, "Bin"],
    ],
    "Good": [
        toy_prebin_table.loc[0, "Good"],
        toy_prebin_table.loc[1:2, "Good"].sum(),
        toy_prebin_table.loc[3, "Good"],
        toy_prebin_table.loc[4, "Good"],
    ],
    "Bad": [
        toy_prebin_table.loc[0, "Bad"],
        toy_prebin_table.loc[1:2, "Bad"].sum(),
        toy_prebin_table.loc[3, "Bad"],
        toy_prebin_table.loc[4, "Bad"],
    ],
})
toy_merged_table, toy_merged_total_iv = calculate_woe_iv_from_counts(toy_merged_counts)

assert toy_merged_table.loc[1, "Count"] == 80
assert toy_merged_table.loc[1, "Good"] == 59
assert toy_merged_table.loc[1, "Bad"] == 21
assert np.isclose(toy_merged_table.loc[1, "Default Rate"], 21 / 80)
assert toy_merged_table["Count"].sum() == toy_prebin_table["Count"].sum() == 200
assert toy_merged_table["Good"].sum() == toy_prebin_table["Good"].sum()
assert toy_merged_table["Bad"].sum() == toy_prebin_table["Bad"].sum()

display(toy_merged_table[["Bin", "Count", "Good", "Bad", "Default Rate", "WOE"]])
print("Adjacent toy merge result: 40 + 40 = 80 observations; merged default rate = 26.25%")
print("Demonstration status: PASS — conceptual adjacent merge only; no real threshold produced")


,Bin,Count,Good,Bad,Default Rate,WOE
0,"(0.999, 40.8]",40.000000,36.000000,4.000000,0.100000,1.686399
1,"(40.8, 80.6] + (80.6, 120.4]",80.000000,59.000000,21.000000,0.262500,0.522189
2,"(120.4, 160.2]",40.000000,20.000000,20.000000,0.500000,-0.510826
3,"(160.2, 200.0]",40.000000,10.000000,30.000000,0.750000,-1.609438


Adjacent toy merge result: 40 + 40 = 80 observations; merged default rate = 26.25%
Demonstration status: PASS — conceptual adjacent merge only; no real threshold produced


## 9. Final-bin count, ordinary sample size, and monotonicity

For 24,000 Train rows, ordinary final-bin guidance is:

- ordinary continuous predictors: usually 3–6 final bins;
- AGE: tendency toward 3–5 final bins;
- preferred ordinary final-bin size: at least 5%, about 1,200 rows;
- 3%–5%, about 720–1,200 rows: acceptable only with justification;
- below 3%: normally too small and should be merged with a reasonable adjacent bin;
- explicit special/business bins may be exceptions and must be visibly flagged for human judgment.

These are design principles, not a mechanical `min_bin_size=5%` algorithm.

**Monotonicity is preferred, not mandatory.** A reasonable overall risk direction is desirable, but small local reversals can be real. Do not over-merge merely to manufacture a perfect monotonic curve. Stability, sample adequacy, interpretability, and reasonable risk structure take priority.

Adjacent merging later considers all six frozen criteria: close default rates, close WOE, small bins, improved stability, business interpretation, and reduction of obvious random fluctuation.


## 10. Candidate special-bin and business-anchor policies

These policies are recorded now but are **not applied to real data in Stage 4.1**.

### BILL_AMT1–BILL_AMT6

- `< 0`: candidate special bin;
- `= 0`: candidate special bin;
- `> 0`: later quantile pre-binning and supervised adjacent merging.

A candidate special bin is not permanently unmergeable. If it is extremely small and its risk behavior/WOE is close to a reasonable neighbor, Stage 4.3 may propose a merge for Human Checkpoint approval.

### PAY_AMT1–PAY_AMT6

- `= 0`: candidate special bin;
- positive values: later quantile pre-binning and supervised adjacent merging.

### current_utilization, avg_utilization, max_utilization

- `= 0`: candidate business anchor;
- `> 1`: candidate business anchor;
- `0 < utilization <= 1`: later supervised binning.

Stage 4.1 does not predefine 25%, 50%, 75%, 80%, 90%, or any other internal utilization cut point.


## 11. PAY, delinquency-summary, and categorical methodology

### PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6

- Treat each month as an ordered discrete credit-behavior status variable; do not quantile-bin it.
- Analyze each month independently by raw state using Count, Good, Bad, Default Rate, WOE, and IV contribution.
- Permit only ordinal-adjacent or coherent-range merging.
- Do not pre-merge `-2/-1/0`; Stage 4.2 must test whether their Train risk-modeling behavior is sufficiently similar.
- Monthly groupings may differ.
- Sparse high-severity states may later support a coherent group such as `3+`, but Stage 4.1 does not decide it.
- Never claim merged raw states have identical business semantics.

### delinquent_months, max_delinquency, severe_delinquent_months

Treat each as a low-cardinality ordinal predictor. Start from each raw integer and allow only adjacent-value merging. A coherent range such as `3+` may later be considered; unordered groups such as `0 + 4` are forbidden.

### Existing categorical predictors

- SEX retains its raw categories.
- EDUCATION_GROUP initially retains Graduate School, University, High School, Other / Unknown.
- MARRIAGE_GROUP initially retains Married, Single, Other / Unknown.

Stage 4.2 first computes category WOE/IV without proactive merging. A merge may only be proposed for extremely sparse support, zero Good/Bad, unstable extreme WOE, or clearly redundant risk behavior, and still requires Human Checkpoint approval.


## 12. Zero-Good / zero-Bad policy

A bin with `Good = 0` or `Bad = 0` produces an infinite raw WOE. The frozen handling order is:

1. **Merge first:** determine whether a small bin can be combined with an adjacent or otherwise logically valid category.
2. **0.5 additive smoothing only as fallback:** use it only if the bin has an explicit reason to remain and no reasonable merge exists.

Every smoothing use must identify the variable and bin explicitly. Smoothing is never silent and is not added to every bin by default.

The next synthetic example demonstrates raw infinity, an adjacent merge that removes it, and the finite arithmetic produced by the 0.5 fallback. It does not create a new production rule.


In [8]:
toy_zero_counts = pd.DataFrame({
    "Bin": ["Toy_Special_A", "Toy_Special_B", "Toy_Ordinary_Neighbor"],
    "Good": [20, 0, 30],
    "Bad": [0, 5, 25],
})
toy_zero_raw, toy_zero_raw_iv = calculate_woe_iv_from_counts(toy_zero_counts)
assert np.isposinf(toy_zero_raw.loc[0, "WOE"])
assert np.isneginf(toy_zero_raw.loc[1, "WOE"])

# Preferred demonstration: merge the two adjacent synthetic special bins.
toy_zero_merged_counts = pd.DataFrame({
    "Bin": ["Toy_Special_A + Toy_Special_B", "Toy_Ordinary_Neighbor"],
    "Good": [20, 30],
    "Bad": [5, 25],
})
toy_zero_merged, toy_zero_merged_iv = calculate_woe_iv_from_counts(toy_zero_merged_counts)
assert np.isfinite(toy_zero_merged["WOE"]).all()

# Fallback demonstration only: add 0.5 to every Good/Bad bin count used in WOE denominators.
toy_zero_smoothed, toy_zero_smoothed_iv = calculate_woe_iv_from_counts(toy_zero_counts, smoothing=0.5)
assert np.isfinite(toy_zero_smoothed["WOE"]).all()
assert np.isfinite(toy_zero_smoothed["IV Contribution"]).all()
assert np.isclose(toy_zero_smoothed.loc[0, "Good for WOE"], 20.5)
assert np.isclose(toy_zero_smoothed.loc[0, "Bad for WOE"], 0.5)
assert np.isclose(toy_zero_smoothed.loc[1, "Good for WOE"], 0.5)
assert np.isclose(toy_zero_smoothed.loc[1, "Bad for WOE"], 5.5)

print("Raw zero-event toy table — infinite WOE is expected:")
display(toy_zero_raw)
print("Preferred merge-first toy result — finite WOE:")
display(toy_zero_merged)
print("Fallback-only 0.5 smoothing arithmetic — finite WOE:")
display(toy_zero_smoothed)
print("Smoothed fallback total IV:", f"{toy_zero_smoothed_iv:.12f}")
print("Zero-event policy demonstration: PASS — merge first; smoothing explicit and fallback-only")


Raw zero-event toy table — infinite WOE is expected:


,Bin,Count,Good,Bad,Default Rate,Dist Good,Dist Bad,WOE,IV Contribution
0,Toy_Special_A,20.000000,20.000000,0.000000,0.000000,0.400000,0.000000,inf,inf
1,Toy_Special_B,5.000000,0.000000,5.000000,1.000000,0.000000,0.166667,-inf,inf
2,Toy_Ordinary_Neighbor,55.000000,30.000000,25.000000,0.454545,0.600000,0.833333,-0.328504,0.076651


Preferred merge-first toy result — finite WOE:


,Bin,Count,Good,Bad,Default Rate,Dist Good,Dist Bad,WOE,IV Contribution
0,Toy_Special_A + Toy_Special_B,25.000000,20.000000,5.000000,0.200000,0.400000,0.166667,0.875469,0.204276
1,Toy_Ordinary_Neighbor,55.000000,30.000000,25.000000,0.454545,0.600000,0.833333,-0.328504,0.076651


Fallback-only 0.5 smoothing arithmetic — finite WOE:


,Bin,Count,Good,Good for WOE,Bad,Bad for WOE,Default Rate,Dist Good,Dist Bad,WOE,IV Contribution
0,Toy_Special_A,20.000000,20.000000,20.500000,0.000000,0.500000,0.000000,0.398058,0.015873,3.221978,1.231392
1,Toy_Special_B,5.000000,0.000000,0.500000,5.000000,5.500000,1.000000,0.009709,0.174603,-2.889490,0.476461
2,Toy_Ordinary_Neighbor,55.000000,30.000000,30.500000,25.000000,25.500000,0.454545,0.592233,0.809524,-0.312546,0.067913


Smoothed fallback total IV: 1.775766475725
Zero-event policy demonstration: PASS — merge first; smoothing explicit and fallback-only


## 13. Future Stage 4.6 Stratified 5-Fold policy — recorded, not executed

Stage 4.2–4.5 will use the complete 24,000-row Train set to freeze bin boundaries, PAY groupings, variable selection, and the model specification.

Stage 4.6 will then use Stratified 5-Fold for internal stability evidence. Across folds, feature engineering, variable selection, bin boundaries, and PAY groupings remain fixed. Within each fold:

1. Re-estimate WOE values using only the approximately 19,200 fold-training rows and the fixed bins.
2. Apply that fold-training WOE mapping to the approximately 4,800 validation rows.
3. Refit Logistic coefficients on the fold-training rows.
4. Use the validation fold only for that fold's AUC/KS and coefficient/WOE stability evidence.

No KFold, Logistic fit, AUC, or KS is executed here.

Methodological limitation: full-Train targets participated indirectly in selecting final boundaries and variables before the stability check. Therefore Stage 4.6 metrics are **internal stability/robustness evidence under a fixed model specification**, not a perfectly unbiased final out-of-sample estimate. The untouched 6,000-row Holdout Test remains sealed until Stage 5 final comparison.


## 14. Frozen methodology summary

- Good/Bad direction: 0 = Good/non-event; 1 = Bad/event.
- WOE: `ln(Dist Good / Dist Bad)`; positive means relatively lower risk.
- IV: sum of `(Dist Good − Dist Bad) × WOE`; descriptive evidence, never an automatic drop judge.
- Continuous workflow: up to about 20 quantile pre-bins, followed by target-aware adjacent merging.
- Ordinary final bins: usually 3–6; AGE tends toward 3–5.
- Ordinary size guidance: preferred ≥5%, 3%–5% with justification, <3% normally merge; special bins may be exceptions.
- Monotonicity: preferred, not mandatory.
- BILL, PAY_AMT, and utilization special-bin/business-anchor policies are frozen as recorded.
- PAY and delinquency summaries use state/value-level analysis and adjacent/coherent-range merging, not quantile binning.
- Existing categorical variables start unchanged.
- Zero-event handling: merge first; explicit 0.5 smoothing only as fallback.
- Future KFold fixes the model specification but re-estimates WOE and coefficients inside each training fold.


In [9]:
assert REAL_PROJECT_DATA_LOADED is False
assert REAL_PREDICTOR_ANALYSIS_PERFORMED is False
assert HOLDOUT_TEST_USED is False
assert VARIABLE_SELECTION_PERFORMED is False
assert MODELING_PERFORMED is False
assert KFOLD_EXECUTED is False
assert np.isclose(programmatic_total_iv, manual_total_iv, rtol=1e-12, atol=1e-12)
assert np.isfinite(toy_zero_smoothed["WOE"]).all()
assert toy_merged_table.loc[1, "Count"] == 80

final_method_status = pd.DataFrame({"Result": [
    "PASS — 0=Good/non-event; 1=Bad/event",
    "PASS — ln(Dist Good / Dist Bad); sign direction frozen",
    "PASS — manual/programmatic WOE and IV agree at 1e-12 tolerance",
    "PASS — synthetic quantile pre-bins and adjacent merge demonstrated",
    "PASS — raw infinity, merge-first, and explicit 0.5 fallback demonstrated",
    "PASS — no real project data, predictor, target, or Holdout Test used",
    "PASS — no variable selection, modeling, KFold, AUC, or KS",
    "PENDING — Architect/user Human Learning Checkpoint",
]}, index=[
    "Good_Bad_convention", "WOE_definition", "WOE_IV_toy_validation",
    "adjacent_merging_toy", "zero_event_policy", "real_data_boundary",
    "prohibited_operations", "next_gate",
])
display(final_method_status)
print("Stage 4.1 method design and toy validation complete; Stage 4.2 not started.")


,Result
Good_Bad_convention,PASS — 0=Good/non-event; 1=Bad/event
WOE_definition,PASS — ln(Dist Good / Dist Bad); sign directio...
WOE_IV_toy_validation,PASS — manual/programmatic WOE and IV agree at...
adjacent_merging_toy,PASS — synthetic quantile pre-bins and adjacen...
zero_event_policy,"PASS — raw infinity, merge-first, and explicit..."
real_data_boundary,"PASS — no real project data, predictor, target..."
prohibited_operations,"PASS — no variable selection, modeling, KFold,..."
next_gate,PENDING — Architect/user Human Learning Checkp...


Stage 4.1 method design and toy validation complete; Stage 4.2 not started.


## 15. Human Learning Checkpoint — mandatory stop

Before Stage 4.2, the Architect and user must confirm understanding of WOE direction, Default Rate versus WOE, IV's non-automatic role, target-aware adjacent merging, pre-bin versus final bin, special bins, preferred-not-mandatory monotonicity, merge-first zero-event handling, fallback smoothing, and fold-specific WOE re-estimation under fixed bins.

**Stop here. Do not analyze a real predictor, open the sealed Holdout Test, perform variable selection, fit a model, run KFold, or begin Stage 4.2.**
